# 31b_baseline

Writes `submission.zip` for the Gemma 4 Developer Agent competition. Run on CPU with internet off, then Submit.

In [ ]:
import json, zipfile
from pathlib import Path

FILES = json.loads("{\n \"agent.yaml\": \"name: swe_baseline_agent\\nmodel: gemma-4-31b-it-qat-w4a16-ct\\ninstruction: !include prompts/system.md\\ntools:\\n  - run_command\\n  - read_file\\n  - edit_file\\n  - write_file\\n  - get_status\\n  - submit_patch\\n  - get_code_neighbors\\n  - search_similar_code\\n  - get_code_subgraph\\n  - agent_tool:\\n      config_path: sub_agents/code_analyzer.yaml\\n      skip_summarization: true\\ngenerate_content_config: !include configs/sampling.yaml\\n\",\n \"configs/sampling.yaml\": \"temperature: 0.2\\ntop_p: 0.95\\nmax_output_tokens: 8192\\nthinking_config:\\n  thinking_budget: 4096\\n  include_thoughts: true\\n\",\n \"eval_config.yaml\": \"# Per-task budgets. The hidden test has ~120 tasks and a 12-hour limit in total.\\n# 8 min / 28 calls matches the public 0.22 config; lower limits scored worse (0.13).\\nevaluation:\\n  timeout_seconds: 240\\n  max_tool_calls: 28\\n  max_time_minutes: 8\\n  max_turns: 80\\n\",\n \"prompts/analyzer.md\": \"You are a specialized code analysis sub-agent. Your role is to inspect repository source files, trace symbol relationships, and identify the root cause of the reported issue.\\n\\n## Instructions\\n1. Use `search_similar_code`, `get_code_neighbors`, `get_code_subgraph`, and `read_file` to locate the exact functions, classes, and line ranges involved in the bug.\\n2. Provide a concise, structured report containing:\\n   - Exact file paths and line numbers to modify.\\n   - Root cause explanation.\\n   - Recommended minimal code change.\\n\",\n \"prompts/system.md\": \"You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\\n\\n## Core Objective: Fast, Minimal, and Precise Fixes\\nAim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8\\u201310 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.\\n\\n## Workflow\\n\\n### 1. Identify Target Files Immediately\\n- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement.\\n- Read only the specific target files and lines using `read_file` or search tools. Do not wander across unrelated files.\\n- If the problem statement does not provide explicit file paths, use `search_similar_code` with keywords from the error message to locate relevant files efficiently, rather than running `find` or `grep` across the entire repo.\\n\\n### 2. Implement the Solution Directly\\n- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.\\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.\\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\\n\\n### 3. Run Targeted Tests Only (Existing Tests May Be Broken)\\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.\\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.\\n- If you need to locate the test file, find it explicitly with `find tests -name \\\"*<name>*.py\\\"` instead of running the test runner across the repo.\\n\\n### 4. Immediate Patch Submission\\n- Once your targeted test passes:\\n  1. Call `submit_patch` immediately.\\n  2. Verify `patch_size > 0` and `files_changed > 0`.\\n  3. Output a short summary of the fix to end the session.\\n\\n## Anti-Patterns to Avoid\\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) \\u2014 always specify the exact test file path.\\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** \\u2014 your task is strictly to implement the fix for the reported issue in source code.\\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.\\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\\n- Do NOT refactor or reformat unrelated functions or files.\\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.\\n\",\n \"sub_agents/code_analyzer.yaml\": \"name: code_analyzer_agent\\ndescription: Analyzes repository source files and symbol graphs to locate root causes.\\nmodel: gemma-4-31b-it-qat-w4a16-ct\\ninstruction: !include ../prompts/analyzer.md\\ntools:\\n  - read_file\\n  - search_similar_code\\n  - get_code_neighbors\\n  - get_code_subgraph\\ngenerate_content_config: !include ../configs/sampling.yaml\\n\"\n}")

out = Path('/kaggle/working/submission.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name, text in FILES.items():
        zf.writestr(name, text)
print(out, out.stat().st_size, 'bytes:', sorted(FILES))
print(FILES['agent.yaml'])
print(FILES['eval_config.yaml'])
